In [ ]:
Q. 1 — Find the Entities
Ans: 
A sensible grouping is:
Customer
customer_id
customer_name
customer_city

Order
order_id
order_date
customer_id

Product
product_id
product_name
unit_price_gbp 

Order Item
order_id
product_id
quantity
customer_id, order_id, and product_id help connect related records.

In [ ]:
Q. 2 — Draw the Relationships
Using the four entities from Q. 1, draw these relationships:
Ans: 
erDiagram
    CUSTOMER ||--o{ ORDER : places
    ORDER ||--o{ ORDER_ITEM : contains
    PRODUCT ||--o{ ORDER_ITEM : appears_in

In [ ]:
Q. 3 — Create the Base Tables
Create MySQL tables for:

In [ ]:
CREATE DATABASE IF NOT EXISTS britmart_modelling;
 
USE britmart_modelling;
 
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    customer_name VARCHAR(100) NOT NULL,
    customer_city VARCHAR(50) NOT NULL
);
 
CREATE TABLE products (
    product_id INT PRIMARY KEY,
    product_name VARCHAR(100) NOT NULL,
    unit_price_gbp DECIMAL(10,2) NOT NULL
);
 
CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT NOT NULL,
    order_date DATE NOT NULL,
    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
);
 
CREATE TABLE order_items (
    order_id INT NOT NULL,
    product_id INT NOT NULL,
    quantity INT NOT NULL,
    PRIMARY KEY (order_id, product_id),
    FOREIGN KEY (order_id)
        REFERENCES orders(order_id),
    FOREIGN KEY (product_id)
        REFERENCES products(product_id)
);

In [ ]:
Q. 4 — MCQ: Entity
Which item is most clearly an entity in this model?
Ans: C. Product

In [ ]:
Q. 5 — Fix a 1NF Problem 
Ans: 
The products column contains multiple values in one cell.

A better row-level design is:

order_id	customer_id	product_id
101	1	    P1
101	1	    P2
102	2	    P3
Each cell now contains one value.

In [ ]:
Q. 6 — Find the 2NF Problem

The key has two parts:
order_id + product_id

But product_name depends only on product_id.
It should be stored with the product:

products
---------
product_id
product_name

The order-item table can then keep:

order_id
product_id
quantity

This removes a partial dependency.

In [ ]:
Q. 7 — Find the 3NF Problem
Ans: 
city depends on postcode, not directly on the customer.

Many customers can share the same postcode, so the city value may be repeated many times.

A possible split is:

customers
---------
customer_id
customer_name
postcode


postcodes
---------
postcode
city


The customer row stores the postcode and the postcode table stores the city.

In [ ]:
Q. 8 — MCQ: Normalisation
What is the main aim of normalisation in an OLTP-style model?
Ans: B. Reduce repeated facts and keep relationships clear.

In [ ]:
Q. 9 — Insert a Small Normalised Dataset

In [ ]:
INSERT INTO customers VALUES
(1, 'Amelia Clarke', 'London'),
(2, 'Oliver Bennett', 'Manchester');
 
INSERT INTO products VALUES
(10, 'Desk Lamp', 25.00),
(11, 'Office Chair', 80.00),
(12, 'Notebook Set', 10.00);
 
INSERT INTO orders VALUES
(1001, 1, '2026-08-01'),
(1002, 2, '2026-08-01'),
(1003, 1, '2026-08-02');
 
INSERT INTO order_items VALUES
(1001, 10, 2),
(1001, 12, 1),
(1002, 11, 1),
(1003, 12, 3);

In [ ]:
Q. 10 — Rebuild a Denormalised Result with a Join

In [ ]:
SELECT
    o.order_id,
    o.order_date,
    c.customer_name,
    c.customer_city,
    p.product_name,
    oi.quantity,
    p.unit_price_gbp,
    oi.quantity * p.unit_price_gbp
        AS line_total_gbp
FROM orders o
INNER JOIN customers c
    ON o.customer_id = c.customer_id
INNER JOIN order_items oi
    ON o.order_id = oi.order_id
INNER JOIN products p
    ON oi.product_id = p.product_id
ORDER BY o.order_id, p.product_id;

In [ ]:
Q. 11 — Compare Update Behaviour
Imagine the product name changes from: Notebook Set to Premium Notebook Set

Ans: Model A needs fewer changes.

The product name is stored once in the product table, so one update changes the main stored value.

In Model B, the same name may appear in many rows. Keeping every repeated copy correct is harder.

In [ ]:
Q. 12 — MCQ: Denormalisation
Why might an analytical dataset deliberately repeat some descriptive values?
Ans: D. To make some analytical queries simpler to read and use.

In [ ]:
Q. 13 — Identify the Fact and Dimensions
Ans: 
A simple star design is:
FactSales
---------
date_key
customer_key
product_key
store_key
quantity
sales_gbp 

Dimensions:
DimDate
DimCustomer
DimProduct
DimStore
The fact table stores the measurable sales event. Dimensions describe the event.

In [ ]:
Q. 14 — Define the Grain
Choose the clearer grain for FactSales. 

Ans:  Choice B is the better grain.

If one order contains several products, one row per order does not preserve product-level detail.

A clear grain statement is: One fact row represents one product line in one order.

In [ ]:
Q. 15 — Draw the Star 
Ans:
flowchart TD
    D["DimDate"] --> F["FactSales"]
    C["DimCustomer"] --> F
    P["DimProduct"] --> F
    S["DimStore"] --> F

In [ ]:
Q. 16 — MCQ: Fact Table
Which values belong most naturally in a sales fact table? 
Ans: A. quantity and sales_gbp

In [ ]:
Q. 17 — Normalise a Poor Order Table 
Ans: 
A suitable design is: 
customers           
---------
customer_id PK
customer_name
customer_city 

products
--------
product_id PK
product_name
unit_price_gbp 

orders
------
order_id PK
customer_id FK
order_date 


order_items
-----------
order_id PK/FK
product_id PK/FK
quantity

In [ ]:
Q. 18 — Find Update Anomalies
In the poor table from Q. 17, Amelia's city appears in two rows. 
Ans: The same customer now appears to have two different cities.

This is an update anomaly caused by repeating the same customer fact across several order rows.

Keeping the customer city in the customer table gives one main place to update it.

In [ ]:
Q. 19 — Convert the Normalised Model into a Sales Star 
Possible design:
Grain:
One row per product line in an order.

FactSales
---------
order_id
date_key
customer_key
product_key
quantity
sales_gbp

DimCustomer
-----------
customer_key
customer_id
customer_name
customer_city

DimProduct
----------
product_key
product_id
product_name 

DimDate
-------
date_key
full_date

In [ ]:
Q. 20 — Review the Model Before Building It 
1. What does one row represent?
2. Which fields identify the row?
3. Which facts repeat?
4. Which fields describe another entity?
5. Which relationships must remain valid?
6. Is the model mainly for transactions or analytics?

Ans: These questions force the team to decide the grain, entities, keys, repeated facts, relationships, and 
main workload before creating tables.

That makes the model easier to understand and reduces accidental duplication or unclear table responsibilities.